# 2. Identity & Compliance Architecture Design

SC-100 asks: **"Design an identity architecture that implements Zero Trust, protects privileged access, and meets compliance requirements."**

## Setup

Make sure you've installed the kernel:
```bash
cd security-certs/sc-100/02-secops-identity-compliance
uv sync
# Notebooks use the local .venv directly -- no global kernel to register.
# In VS Code: open the kernel picker (top-right) and select `.venv`.
# In classic Jupyter: uv run jupyter notebook notebooks/
```
Then pick the **`.venv` kernel** for this folder from the VS Code kernel picker (top-right).

## Glossary (read this first if you're new)

| Term | Plain English |
|------|--------------|
| **Entra ID** | Microsoft's cloud identity provider (formerly "Azure AD") |
| **Conditional Access (CA)** | "If user X from device Y in location Z → allow / require MFA / block" rules |
| **MFA** | Multi-factor authentication. **Phishing-resistant MFA** = FIDO2 key, Windows Hello, or passkey (NOT SMS) |
| **PIM** | Privileged Identity Management — admins activate roles just-in-time instead of holding them 24/7 |
| **PAW** | Privileged Access Workstation — a locked-down laptop used only for admin tasks |
| **Break-glass account** | Emergency admin account for when normal admin sign-in is broken. Cloud-only, phishing-resistant credential kept in a safe, excluded from blocking Conditional Access policies |
| **B2B / CIAM** | Business guests in your workforce tenant / consumer customers in an **external tenant**. Both live under **Microsoft Entra External ID** today; Azure AD B2C is legacy and closed to new tenants |
| **Zero Trust** | "Never trust, always verify" — verify every request regardless of network location |
| **Purview** | Microsoft's suite for **data** governance (labeling, DLP, insider risk) |
| **Azure Policy** | Enforces **infrastructure** rules ("no public storage accounts", etc.) |

**Mental model**: Identity = *who are you?*  Compliance = *what are you allowed to do, and can we prove it?*


## Bad practice → Best practice (identity & compliance)

| ❌ Bad / legacy | ✅ Best practice | Why |
|-----------------|------------------|-----|
| Dozens of permanent Global Admins | **≤5 Global Admins**, rest via **PIM** just-in-time | Shrinks the window an attacker can abuse standing privilege |
| SMS-based MFA everywhere | **Phishing-resistant MFA** (FIDO2 / passkeys / Windows Hello) for admins | SMS can be SIM-swapped or phished via adversary-in-the-middle |
| Per-user MFA toggles (legacy) | MFA enforced via **Conditional Access** policies | Consistent, centrally managed, risk-adaptive |
| Allowing legacy auth (IMAP / POP / SMTP basic) | **Block legacy auth** via CA | Legacy protocols bypass MFA — how most password sprays succeed |
| No break-glass accounts | **Two or more** cloud-only (`*.onmicrosoft.com`) break-glass accounts, each with a **phishing-resistant** credential (FIDO2 passkey or certificate) locked in a safe, **excluded from CA policies that block or restrict sign-in**, permanently active Global Admin in PIM, alerted on in Sentinel, tested every 90 days | You *will* get locked out someday; plan for it — but plan for it *securely*, not by removing MFA |
| Same identity/device for email & admin | **Separate admin accounts + PAW** for control plane | Compromise of the user mailbox no longer compromises the tenant |
| Service principal secrets in code/repos | **Managed identities** + workload identity federation | Zero secrets to leak/rotate |
| "Guests can access anything the link reaches" | **Cross-tenant access policies** + CA scoped to needed apps only | Least privilege for external identities |
| Compliance = yearly audit spreadsheet | **Purview + Defender for Cloud regulatory dashboards + Azure Policy** | Continuous assessment, auto-remediation, evidence export |


## Hands-on: score your Zero Trust maturity

Zero Trust isn't a product — it's a posture across six pillars. The cell below is a simple self-assessment you can edit for your org (or the Fabrikam scenario) and get a weakest-link readout.

In [ ]:
# Zero Trust maturity self-assessment (runnable)
# Microsoft defines 6 pillars — score your org and see the weakest link.
# Each answer: 1 = Traditional, 2 = Advanced, 3 = Optimal.

assessment = {
    "Identity": {
        "MFA enforced for all users":                    1,
        "Conditional Access evaluates every sign-in":    2,
        "Phishing-resistant MFA for admins":             1,
        "Passwordless rollout in progress":              1,
    },
    "Endpoints": {
        "Devices enrolled in MDM (Intune)":              2,
        "Compliance required by Conditional Access":     1,
        "EDR (Defender for Endpoint) on all endpoints":  2,
    },
    "Apps": {
        "App inventory + Defender for Cloud Apps":       2,
        "Apps behind SSO / modern auth only":            2,
        "Legacy auth blocked":                           1,
    },
    "Data": {
        "Sensitivity labels in use":                     1,
        "DLP policies blocking external sharing":        1,
        "Insider risk management deployed":              1,
    },
    "Infrastructure": {
        "Defender for Cloud on all subscriptions":       2,
        "JIT VM access, no public RDP/SSH":              2,
        "Private endpoints for PaaS data services":      1,
    },
    "Network": {
        "Segmentation via NSGs / Azure Firewall":        2,
        "Private access to internal apps (no VPN)":      1,
        "DDoS Network Protection on critical apps" :     1,
    },
}

labels = {1: "Traditional", 2: "Advanced", 3: "Optimal"}
print(f"{'Pillar':<16} {'Score':<10} {'Maturity'}")
print("-" * 55)
overall = []
for pillar, items in assessment.items():
    avg = sum(items.values()) / len(items)
    overall.append(avg)
    bar = "█" * int(avg * 6) + "░" * (18 - int(avg * 6))
    band = labels[round(avg)] if round(avg) in labels else labels[1]
    print(f"{pillar:<16} {avg:.2f}/3    {bar} {band}")

total = sum(overall) / len(overall)
print(f"\nOverall Zero Trust maturity: {total:.2f}/3 ({labels[round(total)]})")

weakest = sorted(assessment.items(), key=lambda kv: sum(kv[1].values()) / len(kv[1]))[:2]
print("\nFocus next on these weakest pillars:")
for pillar, items in weakest:
    gaps = [q for q, v in items.items() if v == 1]
    print(f"  {pillar}:")
    for g in gaps:
        print(f"    - {g}")


## Enterprise Access Model

Microsoft's enterprise access model replaces the legacy "tier model" with a modern approach based on **three planes**:

```
┌────────────────────────────────────────────────────────────────────────┐
│                       CONTROL PLANE                                   │
│  (Highest privilege — protects all other planes)                      │
│                                                                        │
│  • Entra ID Global Admin / Privileged Role Admin                      │
│  • Azure subscription Owner                                           │
│  • Intune Administrator                                               │
│  • Security: PIM, break-glass, PAW devices                            │
├────────────────────────────────────────────────────────────────────────┤
│                       MANAGEMENT PLANE                                │
│  (Manages infrastructure and applications)                            │
│                                                                        │
│  • Azure resource management (Contributor roles)                      │
│  • Microsoft 365 service admins (Exchange, SharePoint)                │
│  • DevOps pipeline admins                                             │
│  • Security: Conditional Access, JIT access, role scoping             │
├────────────────────────────────────────────────────────────────────────┤
│                        DATA / WORKLOAD PLANE                          │
│  (Accesses business data and applications)                            │
│                                                                        │
│  • End users accessing M365, line-of-business apps                    │
│  • Service accounts / workload identities                             │
│  • API consumers                                                       │
│  • Security: MFA, device compliance, app protection                   │
└────────────────────────────────────────────────────────────────────────┘
```

### Key design principle: **No upward access**
- A management plane admin should NOT have control plane access
- A data plane user should NOT be able to escalate to management plane
- Each plane has separate accounts, devices, and Conditional Access policies

In [ ]:
import json

# ===================================================================
# CONDITIONAL ACCESS DESIGN PATTERNS
# An architect designs CA policies as a layered defense system
# ===================================================================

CA_POLICIES = [
    {
        'name': 'CA001 — Require MFA for all users',
        'plane': 'All',
        'users': 'All users (exclude break-glass)',
        'target': 'All cloud apps',
        'conditions': 'Any location, any device',
        'grant': 'Require MFA (authentication strength: phishing-resistant preferred)',
        'priority': 'Critical — baseline for Zero Trust',
    },
    {
        'name': 'CA002 — Block legacy authentication',
        'plane': 'All',
        'users': 'All users',
        'target': 'All cloud apps',
        'conditions': 'Client apps: Exchange ActiveSync, other legacy clients',
        'grant': 'Block access',
        'priority': 'Critical — legacy auth bypasses MFA',
    },
    {
        'name': 'CA003 — Require compliant device for M365',
        'plane': 'Data',
        'users': 'All users',
        'target': 'Office 365',
        'conditions': 'Any location',
        'grant': 'Require device compliance (Intune) OR Microsoft Entra hybrid joined device',
        'priority': 'High — prevents data access from unmanaged devices',
    },
    {
        'name': 'CA004 — Privileged admin: phishing-resistant MFA + compliant device',
        'plane': 'Control',
        'users': 'Directory roles: Global Admin, Privileged Role Admin, Security Admin',
        'target': 'Microsoft Admin Portals',
        'conditions': 'Any location',
        'grant': 'Require phishing-resistant MFA (FIDO2/Windows Hello) + compliant device',
        'priority': 'Critical — control plane must have strongest protection',
    },
    {
        'name': 'CA005 — Block admin access from untrusted locations',
        'plane': 'Control',
        'users': 'Directory roles: All admin roles',
        'target': 'All cloud apps',
        'conditions': 'Location: NOT (corporate network, admin VPN)',
        'grant': 'Block access',
        'priority': 'High — limits admin attack surface',
    },
    {
        'name': 'CA006 — Require app protection for mobile',
        'plane': 'Data',
        'users': 'All users',
        'target': 'Office 365',
        'conditions': 'Device platforms: iOS, Android',
        'grant': 'Require approved client app OR app protection policy',
        'priority': 'Medium — enables BYOD with data protection',
    },
    {
        'name': 'CA007 — Sign-in risk: require MFA',
        'plane': 'All',
        'users': 'All users',
        'target': 'All cloud apps',
        'conditions': 'Sign-in risk: Medium or High',
        'grant': 'Require MFA (cannot use remembered MFA)',
        'priority': 'High — adaptive protection based on risk signals',
    },
    {
        'name': 'CA008 — User risk: require password change',
        'plane': 'All',
        'users': 'All users',
        'target': 'All cloud apps',
        'conditions': 'User risk: High',
        'grant': 'Require password change + MFA',
        'priority': 'High — automated response to compromised credentials',
    },
]

print('=== Conditional Access Policy Design ===\n')
print(f'{"Policy":<60} {"Plane":<12} {"Priority"}')
print('─' * 100)
for p in CA_POLICIES:
    print(f'{p["name"]:<60} {p["plane"]:<12} {p["priority"]}')

print(f'\nTotal policies: {len(CA_POLICIES)}')
print('Design principle: Layer policies from broad (all users) to specific (admin roles).')
print('Always exclude break-glass accounts from blocking policies.')

### How Conditional Access actually decides — the evaluation order

Before you run the simulator, learn the real engine, because SC-100 asks you to *validate* CA designs.

**Step 0 — CA runs *after* the first factor.** The user proves their password (or passwordless credential) to Microsoft Entra ID first. Conditional Access then decides whether that authentication is *enough*. This is why CA can require a second factor but cannot itself replace primary authentication — and why **legacy authentication protocols can only be blocked, never MFA-challenged**: they have no way to present a second factor.

**Phase 1 — Collect and match.** The engine gathers the session details (user/group, target resource, device state and platform, client app, IP/named location, sign-in risk, user risk, insider risk) and evaluates **every** enabled policy to see which ones apply. Assignments are ANDed inside a policy and exclusions always beat inclusions.

**Phase 2 — Enforce.** For every policy that matched:

1. **Block wins. Always.** A single matching Block policy ends the evaluation — no grant control can rescue the sign-in. There is no priority number to reorder this, unlike NSG rules.
2. Otherwise, **all** matching grant controls must be satisfied — policies AND together across the whole set. The effect is "most restrictive wins".
3. *Within* a single policy you choose the operator: **Require all the selected controls** (AND) or **Require one of the selected controls** (OR).
4. Session controls (sign-in frequency, app-enforced restrictions, token protection) are applied last, to the session that was granted.

**Report-only** policies are evaluated and logged but never enforced — which is why break-glass accounts do not need to be excluded from them.

The simulator below implements exactly that: block short-circuits, then the strongest required authentication strength across all matching policies wins.

### Hands-on: run a mini Conditional Access engine

The table above lists what policies *are*. Below we **run** a tiny evaluator so you can see how CA actually makes an allow/block/MFA decision on every sign-in.

In [ ]:
# Mini Conditional Access evaluator (runnable simulator)
# Conditional Access is "if...then" logic evaluated on every sign-in.
# Here is a simplified engine you can read, run, and tweak.
from dataclasses import dataclass
from typing import Callable, List

@dataclass
class SignInContext:
    user: str
    is_admin: bool
    mfa_method: str       # "none", "sms", "authenticator", "fido2"
    device_compliant: bool
    location: str         # "corp", "home", "untrusted"
    sign_in_risk: str     # "none", "low", "medium", "high"
    app: str              # "M365", "AzurePortal", "LegacyEmail"
    break_glass: bool = False

@dataclass
class CAPolicy:
    name: str
    when: Callable[[SignInContext], bool]   # condition
    decision: str                           # "allow", "require_mfa", "require_phishing_resistant_mfa", "block"

POLICIES: List[CAPolicy] = [
    CAPolicy("CA001 Baseline MFA",
             lambda c: not c.break_glass,
             "require_mfa"),
    CAPolicy("CA002 Block legacy auth",
             lambda c: c.app == "LegacyEmail",
             "block"),
    CAPolicy("CA004 Admin: phishing-resistant MFA + compliant device",
             lambda c: c.is_admin and c.app == "AzurePortal",
             "require_phishing_resistant_mfa"),
    CAPolicy("CA005 Block admin from untrusted location",
             lambda c: c.is_admin and c.location == "untrusted",
             "block"),
    CAPolicy("CA007 Sign-in risk medium/high -> MFA",
             lambda c: c.sign_in_risk in ("medium", "high"),
             "require_mfa"),
]

STRENGTH = {"none": 0, "sms": 1, "authenticator": 2, "fido2": 3}

def evaluate(ctx: SignInContext):
    """Return (final_decision, applied_policies).

    Mirrors the real engine: every policy is evaluated, a single Block short-circuits
    everything, and otherwise ALL matching grant controls must be satisfied (so the
    strongest requirement across policies is what the user has to meet).
    """
    if ctx.break_glass:
        return "allow (break-glass)", ["(excluded from CA policies that block or restrict sign-in)"]
    applied, decisions = [], []
    for p in POLICIES:
        if p.when(ctx):
            applied.append(p.name)
            decisions.append(p.decision)
    if "block" in decisions:
        return "BLOCK", applied
    need = 0
    if "require_phishing_resistant_mfa" in decisions:
        need = 3
    elif "require_mfa" in decisions:
        need = 2
    if need == 0:
        return "ALLOW", applied
    if not ctx.device_compliant and "require_phishing_resistant_mfa" in decisions:
        return "BLOCK (admin needs compliant device)", applied
    if STRENGTH[ctx.mfa_method] >= need:
        return "ALLOW (MFA satisfied)", applied
    return f"CHALLENGE: upgrade MFA to strength>={need}", applied

cases = [
    SignInContext("alice", False, "authenticator", True,  "home",      "low",    "M365"),
    SignInContext("bob",   False, "none",          False, "corp",      "none",   "LegacyEmail"),
    SignInContext("admin", True,  "sms",           True,  "corp",      "none",   "AzurePortal"),
    SignInContext("admin", True,  "fido2",         True,  "corp",      "none",   "AzurePortal"),
    SignInContext("admin", True,  "fido2",         True,  "untrusted", "none",   "AzurePortal"),
    SignInContext("eve",   False, "none",          False, "home",      "high",   "M365"),
    # Break-glass still carries a phishing-resistant credential (FIDO2 in a safe). What makes it
    # emergency-proof is the CA EXCLUSION, not the absence of MFA.
    SignInContext("glass", True,  "fido2",         False, "home",      "none",   "AzurePortal", break_glass=True),
]

print("=== Conditional Access evaluation ===\n")
for c in cases:
    decision, applied = evaluate(c)
    tag = "admin" if c.is_admin else "user"
    print(f"{c.user:<6} [{tag}] app={c.app:<12} mfa={c.mfa_method:<13} risk={c.sign_in_risk:<6} loc={c.location:<9} -> {decision}")
    for a in applied:
        print(f"         -> {a}")
    print()

print("Experiment:")
print("  • Flip admin to mfa=\"authenticator\" — see that admin portal needs fido2.")
print("  • Add a policy requiring compliant device for M365.")
print("  • Add a \"block untrusted countries\" policy.")


In [ ]:
# ===================================================================
# PRIVILEGED ACCESS STRATEGY
# Design a complete privileged access protection architecture
# ===================================================================

PRIVILEGED_ACCESS_DESIGN = {
    'PIM (Privileged Identity Management)': {
        'purpose': 'Just-in-time, time-limited role activation',
        'design_decisions': [
            'All admin roles activated through PIM — no permanent assignments except break-glass',
            'Maximum activation duration: 8 hours for management plane, 4 hours for control plane',
            'Require approval for: Global Admin, Privileged Role Admin, Security Admin',
            'Require justification for all activations',
            'MFA required on activation (phishing-resistant for control plane roles)',
            'Access reviews: monthly for control plane, quarterly for management plane',
        ],
    },
    'PAW (Privileged Access Workstations)': {
        'purpose': 'Dedicated, hardened devices for admin tasks',
        'design_decisions': [
            'Tier: Enterprise (for management plane) and Specialized (for control plane)',
            'Managed via Autopilot + Intune with strict compliance policies',
            'No email, web browsing, or personal apps on PAW',
            'Conditional Access: admin portals only accessible from PAW device compliance',
            'Separate user accounts for PAW (admin-username@domain)',
        ],
    },
    'Break-glass accounts': {
        'purpose': 'Emergency access when PIM/MFA/CA are unavailable',
        'design_decisions': [
            'Two or more cloud-only accounts on the *.onmicrosoft.com domain (never synced or federated)',
            'Permanent ACTIVE Global Admin in PIM (never "eligible" - activation could be the thing that is broken)',
            'Excluded from Conditional Access policies that BLOCK or RESTRICT sign-in (report-only policies need no exclusion)',
            'Phishing-resistant credential: FIDO2 passkey or certificate-based auth, in a fireproof safe, split custody',
            'That credential must differ from the method normal admin accounts use (avoid a shared failure mode)',
            'Used only from a designated secure workstation / PAW',
            'Sentinel alert on ANY sign-in from a break-glass account, with a post-mortem review afterwards',
            'Validate at least every 90 days: can they sign in, and did the alert fire?',
            'RETIRED ADVICE: "give break-glass accounts no MFA". Mandatory MFA now covers admin portals, and '
            'resilience comes from the CA exclusion, not from dropping the second factor.',
        ],
    },
    'Workload identities': {
        'purpose': 'Service principals, managed identities for non-human access',
        'design_decisions': [
            'Managed identities preferred over service principals (no credential management)',
            'Workload identity federation for external services (GitHub Actions, Terraform Cloud)',
            'Conditional Access for workload identities (P2 feature)',
            'Regular access reviews for service principal permissions',
            'Monitor for anomalous service principal behavior in Sentinel',
        ],
    },
}

print('=== Privileged Access Architecture ===\n')
for component, details in PRIVILEGED_ACCESS_DESIGN.items():
    print(f'\n{"=" * 70}')
    print(f'{component}')
    print(f'Purpose: {details["purpose"]}')
    print(f'{"=" * 70}')
    for decision in details['design_decisions']:
        print(f'  • {decision}')

## External Identity Design

### When to use which external identity approach:

| Scenario | Solution | Key features |
|----------|----------|------|
| Business partners accessing your apps | **Entra External ID (B2B)** | Guest accounts, cross-tenant access policies |
| Partners with their own Entra ID | **B2B direct connect** | No guest accounts, shared Teams channels |
| Customers using your consumer app | **Entra External ID (CIAM)** | Custom branded sign-up, social login, self-service |
| Verifiable credentials | **Entra Verified ID** | Decentralized identity, privacy-preserving |
| Multi-tenant SaaS application | **Multi-tenant app registration** | Single app serves multiple tenants |

### Cross-tenant access settings (architect's control):

```
YOUR TENANT ◄─── Cross-tenant access policies ───► PARTNER TENANT

Inbound settings (what partners can access in your tenant):
  ✓ Allow B2B collaboration from partner.com
  ✓ Trust partner's MFA (don't re-prompt)
  ✓ Trust partner's compliant device status
  ✗ Block B2B from unknown domains

Outbound settings (what your users can access in partner tenants):
  ✓ Allow users to accept B2B invitations from partner.com
  ✗ Block outbound to consumer tenants (gmail.com, outlook.com)

Tenant restrictions (control which external tenants users can access):
  ✓ Only allow access to approved partner tenants
  ✗ Block access to personal Microsoft accounts from corporate devices
```

In [ ]:
# ===================================================================
# SCENARIO: Design identity architecture for Fabrikam
# ===================================================================

SCENARIO = """
COMPANY: Fabrikam Manufacturing
EMPLOYEES: 20,000 across 15 countries
INDUSTRY: Manufacturing (OT environments)

IDENTITY LANDSCAPE:
  - Entra ID: 20,000 user accounts, synced from 3 AD forests
  - 500 B2B guest accounts (suppliers, contractors)
  - 200 service principals for DevOps pipelines
  - Consumer portal: 50,000 customers (currently using Auth0)
  - 15 Global Admins (too many!)
  - No PIM deployed
  - MFA: only for admins via legacy per-user MFA (not Conditional Access)
  - Conditional Access: 2 policies (basic MFA for admins, block countries)
  - No break-glass accounts documented
  - 3 Entra ID tenants (acquisitions) — no consolidation plan

RECENT INCIDENTS:
  - Contractor account used to access production Azure subscription
  - Service principal with Contributor role leaked in GitHub repo
  - Global Admin account compromised via phishing (password + SMS MFA)

GOALS:
  - Zero Trust identity architecture
  - Reduce admin attack surface
  - Migrate consumer portal to Entra External ID
  - Consolidate tenants where possible
"""

print(SCENARIO)

In [ ]:
# ===================================================================
# IDENTITY ARCHITECTURE DESIGN EVALUATION
# ===================================================================

IDENTITY_DESIGN = {
    'Phase 1 — Immediate (0-30 days): Stop the bleeding': [
        {'action': 'Deploy break-glass accounts (2x cloud-only)', 'risk_reduced': 'Total lockout', 'effort': 'Low'},
        {'action': 'Reduce Global Admins from 15 to 3 (+ 2 break-glass)', 'risk_reduced': 'Lateral escalation', 'effort': 'Medium'},
        {'action': 'Enable PIM for all admin roles', 'risk_reduced': 'Standing privilege abuse', 'effort': 'Medium'},
        {'action': 'Replace SMS MFA with phishing-resistant (FIDO2) for admins', 'risk_reduced': 'MFA bypass attacks', 'effort': 'Medium'},
        {'action': 'Rotate all service principal credentials, move to managed identities', 'risk_reduced': 'Credential leak', 'effort': 'High'},
    ],
    'Phase 2 — Foundation (30-90 days): Build the architecture': [
        {'action': 'Deploy Conditional Access policy set (8 policies from template)', 'risk_reduced': 'Unauthorized access', 'effort': 'Medium'},
        {'action': 'Configure cross-tenant access policies for B2B partners', 'risk_reduced': 'Guest account abuse', 'effort': 'Medium'},
        {'action': 'Deploy PAW devices for control plane admins', 'risk_reduced': 'Admin device compromise', 'effort': 'High'},
        {'action': 'Enable workload identity Conditional Access for service principals', 'risk_reduced': 'SP abuse', 'effort': 'Medium'},
        {'action': 'MFA for all users via Conditional Access (retire per-user MFA)', 'risk_reduced': 'Credential attacks', 'effort': 'Medium'},
    ],
    'Phase 3 — Optimization (90-180 days): Advanced protection': [
        {'action': 'Migrate consumer portal from Auth0 to Entra External ID', 'risk_reduced': 'External identity sprawl', 'effort': 'High'},
        {'action': 'Deploy Entra ID Governance (access reviews, lifecycle workflows)', 'risk_reduced': 'Stale access', 'effort': 'Medium'},
        {'action': 'Implement tenant consolidation (3→1 primary + 1 dev)', 'risk_reduced': 'Multi-tenant complexity', 'effort': 'Very High'},
        {'action': 'Deploy continuous access evaluation (CAE) + token protection', 'risk_reduced': 'Token theft / replay', 'effort': 'Low'},
        {'action': 'Implement Verified ID for supplier onboarding', 'risk_reduced': 'Identity fraud', 'effort': 'High'},
    ],
}

print('=== Identity Architecture Transformation Plan ===\n')
for phase, actions in IDENTITY_DESIGN.items():
    print(f'\n{"=" * 80}')
    print(f'{phase}')
    print(f'{"=" * 80}')
    print(f'{"Action":<65} {"Risk Reduced":<25} {"Effort"}')
    print('─' * 100)
    for a in actions:
        print(f'{a["action"]:<65} {a["risk_reduced"]:<25} {a["effort"]}')

## Compliance Architecture

### Microsoft Purview + Azure Policy + Defender: The compliance triad

```
┌──────────────────────────────────────────────────────────────────────┐
│                     COMPLIANCE ARCHITECTURE                         │
│                                                                      │
│  ┌──────────────────────┐  ┌───────────────────────────────────┐    │
│  │  MICROSOFT PURVIEW   │  │  AZURE POLICY (+ deployment      │    │
│  │                      │  │  stacks / template specs)         │    │
│  │                      │  │                                   │    │
│  │  Data governance:    │  │  Infrastructure governance:       │    │
│  │  • Sensitivity labels│  │  • Enforce resource configs       │    │
│  │  • DLP policies      │  │  • Deny non-compliant deploys    │    │
│  │  • Retention policies│  │  • Auto-remediate drift           │    │
│  │  • Information       │  │  • Regulatory compliance          │    │
│  │    barriers          │  │    built-in initiatives           │    │
│  │  • Records mgmt     │  │  • Custom policy definitions      │    │
│  │  • Insider risk      │  │                                   │    │
│  │  • eDiscovery        │  │  Scope: Management groups →       │    │
│  │  • Communication     │  │  Subscriptions → Resource groups  │    │
│  │    compliance        │  │                                   │    │
│  └──────────────────────┘  └───────────────────────────────────┘    │
│                                                                      │
│  ┌──────────────────────────────────────────────────────────────┐    │
│  │  MICROSOFT DEFENDER FOR CLOUD — Regulatory Compliance       │    │
│  │                                                              │    │
│  │  • Compliance dashboard (NIST 800-53, ISO 27001, PCI DSS)  │    │
│  │  • Continuous assessment of Azure resource compliance        │    │
│  │  • Recommendations mapped to regulatory controls            │    │
│  │  • Export compliance reports for auditors                    │    │
│  └──────────────────────────────────────────────────────────────┘    │
└──────────────────────────────────────────────────────────────────────┘
```

In [ ]:
# ===================================================================
# COMPLIANCE ARCHITECTURE DECISION EXERCISE
# Match the compliance requirement to the right tool
# ===================================================================

COMPLIANCE_REQUIREMENTS = [
    {
        'requirement': 'Prevent Azure resources from being deployed in non-approved regions',
        'tool': 'Azure Policy',
        'specific': 'Built-in policy: "Allowed locations" assigned at management group level',
        'why_not_others': 'Purview is for data governance, not infrastructure. Defender monitors but does not enforce.',
    },
    {
        'requirement': 'Classify and label sensitive customer PII across M365 and Azure',
        'tool': 'Microsoft Purview Information Protection',
        'specific': 'Sensitivity labels with auto-labeling policies + trainable classifiers',
        'why_not_others': 'Azure Policy cannot inspect data content. Defender does not classify data.',
    },
    {
        'requirement': 'Generate a compliance report for PCI DSS auditors showing Azure posture',
        'tool': 'Microsoft Defender for Cloud — Regulatory Compliance',
        'specific': 'Add PCI DSS initiative, export compliance report as PDF',
        'why_not_others': 'Azure Policy enforces rules but does not map to regulatory frameworks with scoring.',
    },
    {
        'requirement': 'Ensure all Azure SQL databases have TDE encryption enabled',
        'tool': 'Azure Policy',
        'specific': 'Built-in policy: "Transparent data encryption should be enabled" with DeployIfNotExists effect',
        'why_not_others': 'This is an infrastructure configuration — Azure Policy can both detect and auto-remediate.',
    },
    {
        'requirement': 'Prevent employees from sharing confidential documents externally via email',
        'tool': 'Microsoft Purview DLP',
        'specific': 'DLP policy: Block external sharing for documents labeled "Confidential" in Exchange Online',
        'why_not_others': 'Azure Policy is for Azure infrastructure. This is M365 data protection.',
    },
    {
        'requirement': 'Detect if an employee is downloading unusually large amounts of data before leaving',
        'tool': 'Microsoft Purview Insider Risk Management',
        'specific': 'Insider risk policy: "Data theft by departing users" triggered by HR connector',
        'why_not_others': 'DLP prevents sharing but does not correlate with HR signals. Insider Risk combines both.',
    },
]

print('=== Compliance Tool Selection Exercise ===\n')
for i, req in enumerate(COMPLIANCE_REQUIREMENTS, 1):
    print(f'\n--- Requirement {i} ---')
    print(f'Need: {req["requirement"]}')
    print(f'Tool: {req["tool"]}')
    print(f'Specific: {req["specific"]}')
    print(f'Why not others: {req["why_not_others"]}')

In [ ]:
# ===================================================================
# IDENTITY & COMPLIANCE ARCHITECTURE QUIZ
# ===================================================================

QUIZ = [
    {
        'question': 'Fabrikam\'s Global Admin account was compromised via phishing (password + SMS MFA).\n'
                    'Which design change BEST prevents this specific attack vector?',
        'options': {
            'A': 'Enable PIM so the admin must activate the role first',
            'B': 'Require phishing-resistant MFA (FIDO2) for admin roles via Conditional Access',
            'C': 'Deploy a PAW device for admin access',
            'D': 'Enable Entra ID Protection sign-in risk policies',
        },
        'answer': 'B',
        'explanation': 'The attack exploited SMS MFA, which is vulnerable to SIM-swap and real-time phishing '
                       '(adversary-in-the-middle). FIDO2 keys are phishing-resistant because they are bound to '
                       'the specific site origin — a phishing page cannot intercept the credential. PIM reduces '
                       'standing access but doesn\'t prevent the phishing. PAW and risk policies help but don\'t '
                       'directly address the MFA weakness.',
    },
    {
        'question': 'Fabrikam has 500 B2B guest accounts from suppliers. A contractor used their guest account\n'
                    'to access a production Azure subscription. What is the BEST architectural fix?',
        'options': {
            'A': 'Delete all B2B guest accounts and use a separate tenant for suppliers',
            'B': 'Configure cross-tenant access policies + Conditional Access requiring compliant devices for guests',
            'C': 'Require all guests to use phishing-resistant MFA',
            'D': 'Enable PIM for all guest accounts',
        },
        'answer': 'B',
        'explanation': 'Cross-tenant access policies let you control what B2B guests can access and trust their '
                       'home tenant\'s MFA/device compliance. Combined with Conditional Access that scopes guest '
                       'access to specific apps (not Azure management), this architecturally prevents production '
                       'access. Option A is too disruptive. C helps but doesn\'t limit scope. D is for privileged '
                       'roles, not general guest access.',
    },
    {
        'question': 'Fabrikam needs to ensure all Azure Storage accounts are encrypted with customer-managed keys\n'
                    'and automatically remediate non-compliant resources. Which tool?',
        'options': {
            'A': 'Microsoft Purview sensitivity labels',
            'B': 'Azure Policy with DeployIfNotExists effect',
            'C': 'Microsoft Defender for Cloud recommendations',
            'D': 'Azure Blueprints',
        },
        'answer': 'B',
        'explanation': 'Azure Policy with DeployIfNotExists can both detect non-compliant storage accounts and '
                       'automatically configure customer-managed key encryption. Defender for Cloud would show '
                       'the recommendation but cannot auto-remediate. Purview is for data classification, not '
                       'infrastructure encryption. Blueprints deployed initial configuration but never enforced '
                       'ongoing compliance -- and Azure Blueprints is being RETIRED (new definitions blocked from '
                       '31 July 2026, full retirement 31 January 2027). Its jobs split into template specs '
                       '(versioned artifacts) and deployment stacks (lifecycle + deny assignments), so it is '
                       'never the right answer to a new design question.',
    },
]

print('=== Identity & Compliance Architecture Quiz ===\n')
for i, q in enumerate(QUIZ, 1):
    print(f'Question {i}:')
    print(f'{q["question"]}\n')
    for key, option in q['options'].items():
        marker = '>>>' if key == q['answer'] else '   '
        print(f'  {marker} {key}. {option}')
    print(f'\n  Answer: {q["answer"]}')
    print(f'  Why: {q["explanation"]}')
    print()

## Real-world mini case study: the Midnight Blizzard lesson

In early 2024 Microsoft disclosed that a test tenant without MFA was password-sprayed, the attacker pivoted to a high-privilege OAuth app, and exfiltrated emails from production. Map it to the controls in this notebook:

| Attacker step | Control that would have caught/blocked it |
|---------------|--------------------------------------------|
| Password spray on a legacy test tenant | **CA001 Baseline MFA** + **CA002 Block legacy auth** |
| OAuth app with excessive permissions | **Entra ID Governance** + app-consent policies + **Defender for Cloud Apps** app discovery |
| Movement from test → production tenant | **Separate tenants with cross-tenant access policies**, no shared privileged accounts |
| Long-lived token abuse | **Continuous Access Evaluation (CAE)** to revoke tokens in near-real-time |

**Architect takeaway**: the weakest identity in your estate sets your security floor. Hunt down "forgotten" tenants, test accounts, and service principals — that's where attackers win.
